In [1]:
print("TEST")

TEST


# Full Summary 
made by Antonijs Svalovs, inspired by the code summary of Georg Tenzing
uh idk

# Libraries

In [2]:
# necessary libraries 
import numpy as np
from numpy.linalg    import norm, solve, matrix_rank

from scipy.special   import roots_jacobi, roots_legendre
from scipy.integrate import quad, solve_ivp
from scipy.optimize  import fsolve
from scipy.linalg    import expm, solve, qr, svd, lstsq, pinv

import matplotlib.pyplot as plt
import sympy as sp

# Useful functions and what they do

## Making arrays

In [8]:
start = 0
stop = 1
num = 100
t, h = np.linspace(start, stop, num, endpoint=False, retstep=True, dtype = np.float64)
""" Returns an array of num evenly spaced values within a given interval [start, stop]
    endpoint: is stop included or not, by default True
    retstep: returns the step size h, by default False
    dtype: data type of the output array, by default None
"""

t = np.arange(start, stop, h, dtype = np.float64) 
""" Returns an array of evenly spaced values within a given interval [start, stop) with step size h"""

a = np.array([1, 2, 3], dtype = complex)
""" Returns an array of complex numbers """
a = np.array([[1, 2, 3], [4, 5, 6]], dtype = complex)
""" Returns a 2D array of complex numbers """
print(a)
a = np.reshape(a, (3, 2))
""" Reshapes the array a to 3x2 """
print(a)
z = np.zeros((3, 3), dtype = complex)
""" Returns a 3x3 array of complex numbers filled with zeros """
o = np.ones((3, 3), dtype = complex)
""" Returns a 3x3 array of complex numbers filled with ones """

z_a = np.zeros_like(a, dtype = complex)
""" Returns a zero array that looks like a -> same shape and type """
o_a = np.ones_like(a, dtype = complex)
""" Returns a one array that looks like a -> same shape and type """

x = np.linspace(0, 2, 5)
y = np.linspace(-1, 1, 5)
X,Y = np.meshgrid(x, y)
""" Returns coordinate matrices from coordinate vectors x and y.
    X contains the x-coordinates and Y contains the y-coordinates of the grid. 
"""
print(X)
print(Y)
# CAREFUL its not x,y like in a coordinate plane, x is the coluns, y is the rows
# If you want to sum over x, you have to sum over columns (axis = 1)
x_sum = np.sum(X, axis = 1)
print(x_sum)

[[1.+0.j 2.+0.j 3.+0.j]
 [4.+0.j 5.+0.j 6.+0.j]]
[[1.+0.j 2.+0.j]
 [3.+0.j 4.+0.j]
 [5.+0.j 6.+0.j]]
[[0.  0.5 1.  1.5 2. ]
 [0.  0.5 1.  1.5 2. ]
 [0.  0.5 1.  1.5 2. ]
 [0.  0.5 1.  1.5 2. ]
 [0.  0.5 1.  1.5 2. ]]
[[-1.  -1.  -1.  -1.  -1. ]
 [-0.5 -0.5 -0.5 -0.5 -0.5]
 [ 0.   0.   0.   0.   0. ]
 [ 0.5  0.5  0.5  0.5  0.5]
 [ 1.   1.   1.   1.   1. ]]
[5. 5. 5. 5. 5.]


## Making Functions
Either you make a function or you use lambda expressions. Lambda expressions are especially useful if you need to pipe a function with multiple inputs into some algorithm / function where the function only has 1 input

In [ ]:
# Example: 1/sin(2x)
def f(x, a):
    return 1/np.sin(a*x)

f = lambda x: f(x, 2)  # a = 2
# OR
a = 2
f = lambda x: 1/np.sin(a*x)

# Now we can use it in solve and other functions
y = solve(f, 0.5, 0.1)

## Broadcasting

Broadcasting means NumPy can apply an operation between arrays of different shapes when their dimensions are compatible. It automatically stretches the smaller array so the operation works, without needing to manually copy values.

Rules:
- Compare shapes from the right.
- A dimension of 1 can be stretched.
- Missing dimensions are treated as 1.
- If shapes are incompatible, NumPy raises an error.

In [2]:
# Simple broadcasting examples
x = np.array([1, 2, 3])
y = x + 5
#print(y)

A = np.array([[1, 2, 3], [4, 5, 6]])
B = A + 10
#print(B)

C = np.array([1, 0, 1])
D = A + C  # C is stretched across each row
#print(D)

# Say I have a 1d array and I want to make a 2d array where i do some operation on the 1d array
t = np.linspace(0, 1, 5)
#I want to multiply each element of t by 2
t = t[:,np.newaxis] * np.array([1,2])
# t[:, np.newaxis] now looks like (5,1) and np.array([1,2]) looks like (2,)
# 5 != 2 so the 2nd axis will be stretched to fit np.array([1,2])
# result will be a (5,2) array
print(t)

[[0.   0.  ]
 [0.25 0.5 ]
 [0.5  1.  ]
 [0.75 1.5 ]
 [1.   2.  ]]


# Chapter 1: Auslöschung, Komplexität
Honestly more important to see how broadcasting works (esp. vandermonde)

In [3]:

# Abl. mit imag Zeitschritt
def diff_ih(f, x, h0):
    maxit = 60 # max # iterations
    h = np.zeros(maxit); h[0] = h0 # Width of the first step
    y = np.zeros(maxit)
    y[0] = np.imag(f(x + 1j*h[0])) / h[0] # First approximation
    
    for k in range(1, maxit):
        h[k] = h[k-1] / 2 # Halve the step size
        y[k] = np.imag(f(x + 1j*h[k])) / h[k] # New approximation
    return y, h

# With Richardson extrapolation
def diffRichardsonV(f,x, h0, rtol=1e-12, atol=1e-12):
    nit = 30 # max depth of iterations
    # first column at once
    h = h0/2**np.arange(nit)
    fp, fm = f(x+h), f(x-h)
    y = (fp-fm)/2/h
    # go column by column in Richardson
    # we prefer to store it in the last part of the vector
    for j in range(1,nit):
        fact = 4**j
        y[j:]=(fact*y[j:] - y[j-1:-1])/(fact-1)
        errest = abs(y[j]-y[j-1])
        print(j, h[j], errest)
        if errest < rtol*abs(y[j]) or errest < atol:
             break
    #return y[:j], h[:j]
    return y[:j+1], h[:j+1] # return the last computed in order to show cancellation
    

# Multiplication with a Diagonalmatrix
def diag_mult(A, d):
    """ Multiplies a matrix A with a diagonal matrix D represented by its diagonal elements d.
        A: 2D numpy array (matrix)
        d: 1D numpy array (diagonal elements of the diagonal matrix)
        Returns: 2D numpy array (result of the multiplication)
    """
    return A * d.diagonal()[:, np.newaxis]  # Use broadcasting to multiply each row of A by the corresponding diagonal element of D

# Efficient construction of Vandermonde Matrices
def vandermonde(t, n):
    """ Constructs a Vandermonde matrix given a vector x and the number of columns n.
        Z: 1D numpy array (input vector)
        n: int (number of columns in the Vandermonde matrix)
        Returns: 2D numpy array (Vandermonde matrix)
    """
    # for k in range t:
    #   Z[:,k] = t**k
    #Z = t[:, np.newaxis]**np.arange(n)
    #print(Z)
    return t[:,np.newaxis]**np.arange(n)
    # takes the input array and turns it into a 2d array (len(t), 1)
    # np.arange(n) is a 1d array of integers from 0 to n-1 -> 1d array (n)
    # since len(t) != n, the 2nd dimension of t[:, np.newaxis] is stretched from 1 to n
    # (only works because the 2nd dim is 1 and the 1st dim is len(t) != n)
    # it now takes the element t[i] and raises it to np.arange(n) -> t[i,:] = t[i]**0, t[i]**1, ..., t[i]**(n-1)

# Chapter 2: Quadrature
How to approximate integration

In [4]:
# Wanna integrate something quickly? -> Use scipy.integrate.quad :)
x1 = -0.54
x2 = 0.12
x3 = 0.78
a = -1; b = 1
f = lambda x: (x-x2)*(x-x3) / ((x1-x2) * (x-x3))
I = quad(f, a, b)[0]  # exact integral

In [ ]:
def quadrature_rule(f, a, b, N):
    x, h = np.linspace(a, b, N + 1, retstep=True)
    xm   = 0.5* (x[1:] + x[:-1]) 
    # Midpoint of each interval: xm = (x[i] + x[i+1]) / 2
    
    I_midp = h       *                                        sum(f(xm))
    # Summierte Mittelpunktregel: Take the midpoint of each interval
    
    I_trap = h / 2.0 * (f(x[0]) + 2.0 * sum(f(x[1:-1]))                  + f(x[-1]))
    # Summierte Trapezregel: Take the average of the endpoints of each interval
    
    I_simp = h / 6.0 * (f(x[0]) + 2.0 * sum(f(x[1:-1])) + 4.0*sum(f(xm)) + f(x[-1]))
    # Summierte Simpsonregel: Take the average of the endpoints and the midpoint of each interval
    I_simp2 = h / 3.0 * sum(f(x[:-2:2]) + 4.0 * sum(f(x[1:-1:2])) + f(x[2::2]))
    # The Gradi way of doing simpson
    
    I_ga2p = 0.5 * h * np.sum(f(xm - h / 2 * np.sqrt(3)) + f(xm +  h / 2 * np.sqrt(3)) )
    # Summierte Gauß-2-Punkte-Regel: Take the midpoint of each interval and add/subtract h/2 * sqrt(3) to get the two Gauss points
    # Yea idk what this is
    
    x13  = x[:-1] + 1/3 * h
    x23  = x[:-1] + 2/3 * h 
    I_sm38 = h / 8.0 * (f(x[0]) + 3.0 * sum(f(x13) + f(x23)) + 2.0 * sum(f(x[1:-1])) + f(x[-1]))
    # I also dont know what this is
    
    # Cotes are evenly spaced nodes
    # 
    c_cotes, w_cotes = compute_weights(a, b, s = 5)
    c_radau, w_radau = radau(s = 5, fixed = "l")
    c_lobat, w_lobat = lobatto(s = 5)
    I_cotes = sum(w_cotes * f(c_cotes))
    I_radau = sum(w_radau * f(c_radau))
    I_lobat = sum(w_lobat * f(c_lobat))
    
    return [I_midp, I_trap, I_simp, I_ga2p, I_cotes, I_radau, I_lobat]

# Example of 2d quadrature using trapezoid
def trap_2d(f, a, b, Nx, c, d, Ny):
    x, dx = np.linspace(a, b, Nx + 1, retstep=True)
    y, dy = np.linspace(c, d, Ny + 1, retstep=True)
    X,Y = np.meshgrid(x, y)
    F = f(X,Y)
    IF_dx = dx / 2.0 * (F[:,0] + 2.0 * np.sum(F[:,1:-1], axis=1) + F[:,-1])
    # IF_dx is the same as 1d TR but we iterate over x. IF_dx(y)
    # axis = 1 means we sum over the columns (x-direction) and keep the rows (y-direction)
    IF_dxdy = dy / 2.0 * (F[0,:] + 2.0 * np.sum(F[1:-1,:], axis=0) + F[-1,:])
    # 1d TR but we use IF_dx instead of F(x,y)
    return IF_dxdy

def radau(s, fixed): 
    # 1 End node is included -> s-1 freely chosen nodes    
    # # This function is for the reference interval [-1, 1], will need to be scaled later
    # jabobi root: alpha: include right end node, beta: include left end node
    ws = 2 / (s**2) # Radau weights at -1 or 1, cannot be calculated with jacobi weights
    if fixed == 'r':
        nodes, w = roots_jacobi(s - 1, alpha = 1, beta = 0)  # Jacobi nodes and weights 
        c = np.hstack([nodes, 1])                        # Radau nodes
        b = np.hstack([(w / (1 - nodes)) , ws])          # Radau weights (1-t)**1 * (1+t)**0
    elif fixed == 'l':
        nodes, w = roots_jacobi(s - 1, alpha = 0, beta = 1)  # Jacobi nodes and weights
        c = np.hstack([-1, nodes])                       # Radau nodes
        b = np.hstack([ws, (w / (1 + nodes))])           # Radau weights (1-t)**0 * (1+t)**1
    return c, b

def lobatto(s): 
    #Both end nodes are included -> s-2 freely chosen nodes
    # This function is for the reference interval [-1, 1], will need to be scaled later
    # Jabobi root: alpha: include right end node, beta: include left end node
    inside_nodes, w = roots_jacobi(s - 2, alpha=1, beta=1) # Jacobi nodes and weights
    c = np.hstack([-1, inside_nodes, 1])                   # Lobatto nodes
    w1 = ws = 2 / ((s - 1) * s)                            # Lobatto weights at -1 and 1                         
    b = np.hstack([w1, w / (1 - inside_nodes**2), ws])     # Lobatto weights [-1,1] (1-t)**1 * (1+t)**1
    return c, b


def transform_quadrature_interval(nodes, weights, a, b):
    transformed_nodes   = 0.5 * (b - a) * nodes + 0.5 * (a + b)
    transformed_weights = 0.5 * (b - a) * weights
    return transformed_nodes, transformed_weights

def compute_weights(a, b, s, x=None): # Page: not 10
    c   = np.linspace(a, b, s)                            # equidistant nodes 
    c   = x                                               # if nodes are given

    # V = np.array([c**i for i in range(s)])                 
    V   = np.vander(c, N=s, increasing=True).T            # Vandermonde matrix of the nodes c
    rhs = np.array([(b**(k + 1) - a**(k + 1)) / (k + 1)   # exact integral of monomials x^p from a to b
                    for k in range(s) ])
    
    w   = np.linalg.solve(V, rhs)                         # w = M^{-1} * rhs   
    return c, w

def compute_weights_and_nodes(): # if both nodes and weights are not given
    def equations(vars):
        x1, x2, w1, w2 = vars
        rhs = [1, 0, 1/3, 0] # [-1,1]
        # rhs = [1, 1/2, 1/3, 1/4] # if [a,b] = [0,1]
        return [w1 * x1**i + w2 * x2**i - rhs[i] for i in range(len(rhs))]
    
    return fsolve(equations, [1, 1, 1, 1])

def compute_errors_and_order(integrand, a, b, I_ref, quadrature): # Page: 17 
    n_evals = 2.0 ** np.arange(3, 10)

    errors = np.array([abs(I_ref - quadrature(integrand, a, b, n)) for n in n_evals]) 
    order  = - np.polyfit(np.log(n_evals), np.log(errors), deg=1)[0]

    plt.loglog(n_evals, errors,         label=quadrature.__name__)
    plt.loglog(n_evals, n_evals**order, label=r"$n^{-2.0}$")
    return errors, order

def compute_order_and_genauigkeitsgrad(method, n, tol = 1e-11): 
    a, b = -1, 1 # Interval for the quadrature rule
    c, w = method(n)
    max_degree = 2 * n
    for k in range(max_degree + 1):
        f_ref = lambda x: x ** k  
        I_exact  = (b**(k + 1) - a**(k + 1)) / (k + 1)  # = quad(f, a, b)[0] if [a,b] =! [-1,1]
        I_approx = np.sum(w * f_ref(c))
        if abs(I_approx - I_exact) > tol:
            return k, k-1 

def gauss(f, a, b, N):
    x, h = np.linspace(a, b, N + 1, retstep=True)
    [nodes, weights] = roots_legendre(5) # 5 nodes
    # Gauss quadrature rule (we just use the nodes and weights for the entire interval)
    n_scaled = 0.5 * (b - a) * nodes + 0.5 * (a + b)
    w_scaled = 0.5 * (b - a) * weights
    I_gaus   = np.dot(w_scaled, f(n_scaled))
    
    ni_scaled = 0.5 * h * nodes + 0.5 * (x[:-1] + x[1:])  # nodes mapped to each subinterval
    w_scaled  = 0.5 * h * weights                          # the weights are the same for each subinterval (provided h the same)
    I_comp    = np.sum([np.dot(w_scaled, f(ni_scaled[i])) for i in range(N)])  # Composite Gauss quadrature rule
    return [I_gaus, I_comp]

# Unknown quadrature formula, just given nodes and weights:
def Q(f, x, w):
    """Approximate quadrature rule.
    INPUT:
        f         : Function
        x         : nodes of the quadrature
        w         : weights of the quadrature
    """
    return np.sum(w * f(x))

def composite_Q(f, a, b, N, xn, wn):
    """Approximate quadrature rule.
    INPUT:
        f         : Function
        a         : left point of integration
        b         : right point of integration
        N         : total number of sub intervals
        xn         : nodes of the quadrature
        wn         : weights of the quadrature
    """
    I = 0.0
    x, h = np.linspace(a, b, N + 1, retstep=True)
    for xi in x[:-1]:  # run over the intervals
        ts = 0.5 * h * xn + xi + 0.5 * h  # nodes mapped to [xi,xi+h]
        I += Q(f, ts, wn) * 0.5 * h

    return I

# Chapter 3: Trigonometrische Interpolation


In [ ]:
def evaliptrig_v1(y,N):
    """" take a trig polynomial y of length n and evaluate it at N points
         only works for even n and real coeficients"""
    n = len(y)                              # Number of input samples (assumed periodic samples of a function)
    if (n%2) == 0: 
       c = np.fft.ifft(y)                   # Compute Fourier coefficients (inverse FFT of sample values)  
       a = np.zeros(N, dtype=complex)       # Initialize zero-padded array for upsampling (length N)
       a[:n//2] = c[:n//2]                  # Copy first half of Fourier coefficients into beginning of a
       a[N-n//2:] = c[n//2:]                # Copy second half into end of a (preserves symmetry / periodicity)
       v = np.fft.fft(a)                    # Compute FFT of padded spectrum → evaluates interpolated values
       return np.real(v)
    else: raise(TypeError, 'odd length')

def evaliptrig_v2(y,N):
    """" take a trig polynomial y of length n and evaluate it at N points
         works for any coefficients but we need to scale fft, ifft"""
    n = len(y)                              # Number of input samples (assumed periodic samples of a function)
    if (n%2) == 0: 
       c = np.fft.fft(y) * 1./n             # Compute Fourier coefficients (inverse FFT of sample values)  
       a = np.zeros(N, dtype=complex)       # Initialize zero-padded array for upsampling (length N)
       a[:n//2] = c[:n//2]                  # Copy first half of Fourier coefficients into beginning of a
       a[N-n//2:] = c[n//2:]                # Copy second half into end of a (preserves symmetry / periodicity)
       v = np.fft.ifft(a) * N               # Compute FFT of padded spectrum → evaluates interpolated values
       return np.real(v)
    else: raise(TypeError, 'odd length')
    
def evaliDtrig(y,N):
    """ take a trig polynomial y of length n and evaluate its derivative at N points
         only works for even n and real coeficients
         Its similar for v2"""
    n = len(y)                              # Number of input samples (assumed periodic samples of a function)
    if (n%2) == 0: 
       c = np.fft.ifft(y)                   # Compute Fourier coefficients (inverse FFT of sample values)
       # THE DERIVATIVE HAPPENS HERE
       k = np.fft.fftfreq(n) * n            # frequencies
       dc = -2j * np.pi * k * c             # derivative in Fourier space   
       # OK WE DERIVED NOW WE ZERO PAD
       a = np.zeros(N, dtype=complex)       # Initialize zero-padded array for upsampling (length N)
       a[:n//2] = dc[:n//2]                 # Copy first half of Fourier coefficients into beginning of a
       a[N-n//2:] = dc[n//2:]               # Copy second half into end of a (preserves symmetry / periodicity)
       v = np.fft.fft(a)                    # Compute FFT of padded spectrum → evaluates interpolated values
       return np.real(v)
    else: raise(TypeError, 'odd length')
    
    
def convtrig(f,N=2**15,nmax=1+2**7):
    tt = np.linspace(0,1,N,endpoint=False)  # reference points to evaluate
    f_exact = f(tt)                         # evaluate at reference points
        
    n = 2
    verror = []
    vn = []
    while n < nmax:
        t = np.linspace(0,1,n,endpoint=False )  # n interpolation points between 0 and 1
        y = f(t)                                # evaluate the function on these points
                                                # evaluate the trig. interpolant in N points
        f_interp = evaliptrig_v1(y,N) #its only for real functions so we use v1
        d = abs(f_interp-f_exact); 
        error = d.max() 
        verror += [error] # Append the error to the list of errors
        vn += [n] # Append the current n to keep track of which error belongs to which n
        n += 4 # Increment n by 4 for the next iteration (to keep n even)
        
    plt.plot(tt, f_exact, label='Exact function')
    plt.plot(tt, f_interp, label='Interpolated function')
    plt.xlabel('x')
    plt.ylabel('f(x)')
    plt.legend()
    return vn, verror

# def plot_fourier_interpolation():
#     N_ref = 4096                                             # High-resolution number of evaluation points
#     n_samples = 128                                          # Low-resolution Number of sample points for interpolation

#     fine_gird   = np.linspace(0, 1, N_ref, endpoint=False)   # High-res grid over [0, 1] with N points (no endpoint)
#     x_coarse = np.linspace(0, 1, n_samples, endpoint=False)  # Generate n equally spaced points on [0,1)

#     f_exact = f(fine_gird)                                   # Evaluate exact step function on fine grid

#     f_samples = f(x_coarse)                                  # Evaluate f at those interpolation points
#     f_interp = evaliptrig(f_samples, N_ref)                  # Evaluate trigonometric interpolant on fine grid and take real part
        
#     plt.plot(fine_gird, f_exact)                             # Plot exact         function (fine resolution)
#     plt.plot(fine_gird, f_interp)                            # Plot interpolated  function (coarse resolution)                      

def plot_fourier_coeffs():
    n_samples = 2**10                                                # Number of points to sample the function (grid size)
    sample_points, h = np.linspace(0, 1, n_samples, endpoint=False, retstep=True)     # Uniform sampling points in [0, 1)

    f_samples                  = f(sample_points)                    # Evaluate function at those sample points
    fourier_coeffs             = np.fft.ifft(f_samples)              # Compute Fourier coefficients (via IFFT)
    fourier_coeffs_centered    = np.fft.fftshift(fourier_coeffs)     # Center Fourier coefficients
    fourier_magnitude_centered = abs(fourier_coeffs_centered)        # Magnitudes of Fourier coefficients 

    half_n_samples = n_samples / 2
    # freq_indices_centered_1 = np.arange(- half_n_samples, half_n_samples)   # Frequency index axis (centered around 0)
    freq_indices_centered_2 = np.fft.fftshift(np.fft.fftfreq(n_samples, d=h)) # h = 1.0/n_samples

    plt.semilogy(freq_indices_centered_2, fourier_magnitude_centered)  

def filter_power_spectrum(f, cutoff_ratio, n_samples=2**10):
    sample_points = np.linspace(0, 1, n_samples, endpoint=False)                    # Uniform sampling points in [0, 1)
    f_samples = f(sample_points)                                                    # Evaluate function at those sample points
    fourier_coeffs = np.fft.fft(f_samples)                                          # Compute Fourier coefficients (via IFFT)
    
    power_spectrum = np.abs(fourier_coeffs)**2                                      # Compute power spectrum (magnitude squared of Fourier coefficients)
    max_power = np.max(power_spectrum)                                              # Maximum power in the spectrum
    
    filtered_coeffs = np.where(power_spectrum < cutoff_ratio * max_power, 
                               0., fourier_coeffs)                                  # Zero out coefficients below cutoff ratio
    
    f_filtered = np.real(np.fft.ifft(filtered_coeffs))                              # Inverse FFT to get filtered function in time domain
    power_spectrum_filtered = np.abs(filtered_coeffs)**2/n_samples                  # Compute power spectrum of filtered coefficients
    
    
    return f_filtered, power_spectrum, power_spectrum_filtered